#### bird_observation + farm_master → farm_bird_exposure_features (농장 × 철새)
##### grain: farm_id + base_date
##### H3 격자 인덱스로 반경 필터 최적화 (cross join 대신 equi-join)
##### 반경: 1, 3, 5, 7, 10 km

In [ ]:
# H3 (Uber 육각형 격자 인덱스): 지구를 육각형 셀로 나눠서 각 위경도에 셀 ID를 부여
%pip install h3
%pip install h3-pyspark


In [ ]:
import sys
sys.path.append("/Workspace/방역로/00_Shared_Utils")
from utils_config import CATALOG
 
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from functools import reduce
import h3

In [ ]:
# silver.bird observation 현재 데이터 바꼈습니다. ml학습용이면 ml bird observation 써야함!
BIRD_TABLE = f"{CATALOG}.silver.bird_observation"
H3_RES     = 7
RADII_KM   = [1, 3, 5, 7, 10]

# # 자동파이프라인 학습용!
CONFIGS = [
    (
        f"{CATALOG}.silver.farm_master",
        # 출력 테이블
        f"{CATALOG}.gold.farm_bird_exposure_features_daily",
        [7, 30],
    ),
]

In [ ]:
# H3 UDF 정의
 
@F.udf("string")
def to_h3(lat, lon):
    if lat is None or lon is None:
        return None
    return h3.latlng_to_cell(lat, lon, H3_RES)
 
@F.udf("array<string>")
def get_neighbors(h3_idx, k):
    """반경 k링 이내 셀 목록 반환 (자기 자신 포함)"""
    if h3_idx is None:
        return None
    # AS-IS: h3.k_ring(h3_idx, k)
    # TO-BE: h3.grid_disk로 변경
    return list(h3.grid_disk(h3_idx, k))

def km_to_kring(km):
    return max(1, int(km / 1.2) + 1)

In [ ]:
# 키워드 (taxon_nm 실제값 기준)
 
DUCK_SPECIES  = ["가창오리","고방오리","넓적부리","쇠오리","아메리카홍머리오리",
                 "알락오리","원앙","청둥오리","청머리오리","혹부리오리","홍머리오리","황오리","흰뺨검둥오리"]
GOOSE_SPECIES = ["쇠기러기","큰기러기","캐나다기러기"]

duck_cond  = F.col("taxon_nm").isin(DUCK_SPECIES)
goose_cond = F.col("taxon_nm").isin(GOOSE_SPECIES)

In [ ]:
# 반경 × 기간 집계 함수
 
def agg_by_radius(df, radius_km, periods):
    filtered = df.filter(F.col("dist_km") <= radius_km)
    r = str(radius_km)
    aggs = []
    for p in periods:
        aggs += [
            # 관측 레코드 전체 count (동일 개체 여러 시점 각각 카운트)
            F.count(F.when((F.col("days_diff") >= 0) & (F.col("days_diff") < p), F.lit(1))).alias(f"bird_obs_count_{p}d_{r}km"),
            # 이동거리 합계 — 첫 관측점 NULL은 0으로 처리
            F.coalesce(
                F.sum(F.when((F.col("days_diff") >= 0) & (F.col("days_diff") < p),
                             F.coalesce(F.col("distance_from_prev_km"), F.lit(0.0)))),
                F.lit(0.0)
            ).alias(f"bird_distance_sum_{p}d_{r}km"),
            # 오리류/기러기류 관측 수
            F.count(F.when((F.col("days_diff") >= 0) & (F.col("days_diff") < p) & duck_cond,  F.lit(1))).alias(f"duck_obs_count_{p}d_{r}km"),
            F.count(F.when((F.col("days_diff") >= 0) & (F.col("days_diff") < p) & goose_cond, F.lit(1))).alias(f"goose_obs_count_{p}d_{r}km"),
        ]
    return filtered.groupBy("farm_id", "reference_date").agg(*aggs)

In [ ]:
def load_farm(table_path):
    df = (
        spark.read.table(table_path)
        # .withColumn("farm_id", F.md5(F.concat_ws("|", F.col("farm_name"), F.col("address"))))
        .withColumn("coord_is_null", F.col("latitude").isNull() | F.col("longitude").isNull())
        .withColumn("farm_h3", to_h3(F.col("latitude"), F.col("longitude")))
        .withColumn("neighbor_cells", get_neighbors(F.col("farm_h3"), F.lit(km_to_kring(10))))
        .dropDuplicates(["farm_id", "reference_date"])
    )
    null_count = df.filter(F.col("coord_is_null")).count()
    if null_count > 0:
        print(f"[WARNING] 좌표 NULL 농장 {null_count}건")
    return df

In [ ]:
# 메인 루프
 
for csv_path, output_table, periods in CONFIGS:
    try:
        max_period = max(periods)
        print(f"\n{'='*60}")
        print(f"처리: {output_table} | 기간: {periods}일")
 
        farm_df = load_farm(csv_path)
 
        # 철새 데이터 매 루프마다 새로 로드 (컬럼명 오염 방지)
        bird_df = (
            spark.read.table(BIRD_TABLE)
            .select(
                "recv_id", "taxon_nm", "season_kst",
                "observed_date_kst",
                "longitude", "latitude",
                "distance_from_prev_km",
            )
            .withColumn("bird_h3", to_h3(F.col("latitude"), F.col("longitude")))
        )
 
        # H3 equi-join — 최대 반경 10km 안 관측점만 조인 (성능 최적화)
        joined = (
            farm_df
            .select("farm_id", "reference_date", "neighbor_cells",
                    F.col("latitude").alias("farm_lat"),
                    F.col("longitude").alias("farm_lon"))
            .withColumn("cell", F.explode("neighbor_cells"))
            .drop("neighbor_cells")
            .join(bird_df.withColumnRenamed("bird_h3", "cell"), on="cell", how="inner")
            # Haversine 정밀 거리 계산
            .withColumn(
                "dist_km",
                F.degrees(
                    F.acos(
                        F.least(F.lit(1.0),
                            F.sin(F.radians(F.col("farm_lat"))) * F.sin(F.radians(F.col("latitude"))) +
                            F.cos(F.radians(F.col("farm_lat"))) * F.cos(F.radians(F.col("latitude"))) *
                            F.cos(F.radians(F.col("farm_lon") - F.col("longitude")))
                        )
                    )
                ) * F.lit(111.195)
            )
            .filter(F.col("dist_km") <= 10.0)
            # reference_date 기준 경과일 (당일 미포함: days_diff >= 0 & < p)
            .withColumn("days_diff", F.datediff(F.col("reference_date"), F.col("observed_date_kst")))
            .filter((F.col("days_diff") >= 0) & (F.col("days_diff") <= max_period))
            .cache()
        )
 
        # 반경별 집계 → wide format join
        radius_dfs = [agg_by_radius(joined, r, periods) for r in RADII_KM]
        agg_result = reduce(lambda a, b: a.join(b, on=["farm_id", "reference_date"], how="outer"), radius_dfs)
 
        # 전체 농장 기준 left join
        # 좌표 정상 & 관측 없음 → 0 / 좌표 NULL → NULL 유지
        
        #--------------------
        # 테이블용
        
        farm_base = (
            spark.read.table(csv_path)  # csv_path가 이제 테이블명
            # .withColumn("farm_id", F.md5(F.concat_ws("|", F.col("farm_name"), F.col("address"))))
            .withColumn("coord_is_null", F.col("latitude").isNull() | F.col("longitude").isNull())
            .select("farm_id", "reference_date", "coord_is_null")
            .dropDuplicates(["farm_id", "reference_date"])
)
        #-------------------
 
        feat_cols = [c for c in agg_result.columns if c not in ["farm_id", "reference_date"]]
        result = farm_base.join(agg_result, on=["farm_id", "reference_date"], how="left")
 
        # 좌표 정상이면 NULL → 0, 좌표 NULL이면 NULL 유지
        for col_name in feat_cols:
            result = result.withColumn(
                col_name,
                F.when(~F.col("coord_is_null"), F.coalesce(F.col(col_name), F.lit(0)))
                .otherwise(F.lit(None))
            )
        result = result.drop("coord_is_null")
 
        # 3km 고정 피처 (species_count, dominant_species)
        filtered_3km = joined.filter(F.col("dist_km") <= 3.0)
 
        species_aggs = [
            F.countDistinct(F.when((F.col("days_diff") >= 0) & (F.col("days_diff") < p), F.col("taxon_nm"))).alias(f"species_count_{p}d_3km")
            for p in periods
        ]
        species_df = filtered_3km.groupBy("farm_id", "reference_date").agg(*species_aggs)
 
        dominant_df = (
            filtered_3km.filter((F.col("days_diff") >= 0) & (F.col("days_diff") < max_period))
            .groupBy("farm_id", "reference_date", "taxon_nm")
            .agg(F.count("recv_id").alias("cnt"))
            .withColumn("rn", F.row_number().over(
                Window.partitionBy("farm_id", "reference_date").orderBy(F.col("cnt").desc())
            ))
            .filter(F.col("rn") == 1)
            .select("farm_id", "reference_date", F.col("taxon_nm").alias(f"dominant_species_{max_period}d_3km"))
        )
 
        result = (
            result
            .join(species_df,  on=["farm_id", "reference_date"], how="left")
            .join(dominant_df, on=["farm_id", "reference_date"], how="left")
            .withColumn("created_at", F.current_timestamp())
        )

 
        spark.sql(f"DROP TABLE IF EXISTS {output_table}")
        (
            result.write.format("delta")
            .mode("overwrite")
            .partitionBy("reference_date")
            .saveAsTable(output_table)
        )
        joined.unpersist()
        print(f"완료 → {output_table}")
 
    except Exception as e:
        print(f"[FAIL] {output_table}: {e}")
 

In [ ]:
%sql
select * from dt4_team1_databricks.gold.farm_bird_exposure_features_daily limit(10)